In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 08:59:10.789538: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 08:59:11.643103: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {}
      
    },
  "temp_data_path": "../../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 08:59:12,786 [DEBUG] [Rain] Rain is initialized
2023-07-04 08:59:12,788 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 08:59:12,789 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 08:59:12,790 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-04 08:59:12,791 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 08:59:12,792 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 08:59:12,794 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 08:59:12,795 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 08:59:12,797 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 08:59:12,798 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 08:59:12,804 [DEBUG] [Rain] Creating workers
2023-07-04 08:59:12,812 [INFO] [Provisioner] provisioner is serving
2023-07-04 08:59:12,813 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 08:59:12,815 [INFO] [Coordinator] coordinator is serving
2023-07-04 08:59:12,816 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 08:59:12,822 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 08:59:12,823 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 08:59:12,824 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 08:59:12,826 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:59:12,830 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 08:59:12,831 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:59:12,833 [INFO] [W

157/157 [==============================] - 2s 6ms/step - loss: 0.7129 - accuracy: 0.7754


2023-07-04 08:59:30,032 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-04 08:59:30,036 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-04 08:59:30,037 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-04 08:59:30,037 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to coordinator
sending data to coordinator


2023-07-04 08:59:30,302 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-04 08:59:30,304 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 08:59:30,324 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
2023-07-04 08:59:30,326 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-04 08:59:30,405 [DEBUG] [DeepLearning] Iteration 1/3 complete for worker 2.
2023-07-04 08:59:30,406 [DEBUG] [DeepLearning] Iteration 1/3 complete for worker 1.
2023-07-04 08:59:30,407 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-04 08:59:30,408 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-04 08:59:30,409 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-04 08:59:30,411 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-04 08:59:30,411 [DEBUG] [DividerAmbassador] Sending ../../../..//RainData/divider/2.pkl to worker2
2023-

...

2023-07-04 08:59:31,108 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 08:59:31,111 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-04 08:59:31,112 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-04 08:59:31,113 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker1
2023-07-04 08:59:31,114 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker2
2023-07-04 08:59:31,115 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-04 08:59:31,116 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-04 08:59:31,116 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-04 08:59:31,205 [DEBUG] [DeepLearning] Iteration 1/3 complete for worker 3.
2023-07-04 08:59:3

Error in loading the data: Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
 Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 08:59:31,493 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-04 08:59:31,495 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker3
2023-07-04 08:59:31,496 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3
2023-07-04 08:59:31,550 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-04 08:59:31,553 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 08:59:31,564 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-04 08:59:31,565 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-04 08:59:31,582 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-04 08:59:31,584 [DEBUG] [DividerAmbassador] divider begins 

Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 08:59:31,844 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-04 08:59:31,846 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-04 08:59:31,895 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 08:59:31,898 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-04 08:59:31,920 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
2023-07-04 08:59:31,921 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-04 08:59:31,922 [DEBUG] [Divider] Divider stopped serving
2023-07-04 08:59:31,924 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-04 08:59:31,924 [DEBUG] [Divider] Divider stopped serving
2023-07-04 08:59:31,925 [INFO] [Provisioner] provisioner stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 1.2448 - accuracy: 0.8434

Test accuracy: 84.3%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 08:59:32,233 [DEBUG] [Rain] Creating workers
2023-07-04 08:59:32,235 [INFO] [Provisioner] provisioner is serving
2023-07-04 08:59:32,236 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 08:59:32,237 [INFO] [Coordinator] coordinator is serving
2023-07-04 08:59:32,237 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 08:59:32,239 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 08:59:32,240 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 08:59:32,241 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 08:59:32,242 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:59:32,244 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 08:59:32,244 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 08:59:32,245 [DEBUG] [TemporaryFilesManager] Creating temp

Error in loading the data:  Ran out of input
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 08:59:46,826 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-04 08:59:46,827 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker2
2023-07-04 08:59:46,831 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-04 08:59:46,832 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker3
2023-07-04 08:59:46,832 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-04 08:59:46,832 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-04 08:59:46,834 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 2
2023-07-04 08:59:46,834 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-04 08:59:46,835 [DEBUG] [DividerAmbassa

Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 08:59:47,278 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-04 08:59:47,278 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-04 08:59:47,278 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-04 08:59:47,279 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker3
2023-07-04 08:59:47,280 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker2
2023-07-04 08:59:47,281 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker1
2023-07-04 08:59:47,281 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3
2023-07-04 08:59:47,282 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 3
2023-07-04 08:59:47,281 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3
2023-07-04 08:59:47

Error in loading the data: Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
 Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in receiving the data:  'NoneType' object is not subscriptable
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 08:59:47,704 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-04 08:59:47,705 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 08:59:47,705 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-04 08:59:47,725 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-04 08:59:47,727 [DEBUG] [DeepLearning] Iteration 3/3 complete.
2023-07-04 08:59:47,728 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-04 08:59:47,729 [DEBUG] [Divider] Divider stopped serving
2023-07-04 08:59:47,730 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-04 08:59:47,731 [DEBUG] [Divider] Divider stopped serving
2023-07-04 08:59:47,731 [INFO] [Provisioner] provisioner stopped serving


In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 1.2448 - accuracy: 0.8434

Test accuracy: 84.3%


2023-07-04 09:03:49,376 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-04 09:03:49,376 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 1
2023-07-04 09:03:50,707 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3
2023-07-04 09:03:50,707 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 3
INFO:Worker_50153:Running the worker with id: 3 on iteration: 3


 18/157 [==>...........................] - ETA: 0s - loss: 0.8338 - accuracy: 0.7656

157/157 [==============================] - 2s 5ms/step - loss: 0.4627 - accuracy: 0.8607
sending data to coordinator


2023-07-04 09:04:00,489 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-04 09:04:00,489 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
INFO:Worker_50152:Running the worker with id: 2 on iteration: 2


...

Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 09:04:20,186 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
2023-07-04 09:04:20,186 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
INFO:Worker_50151:Running the worker with id: 1 on iteration: 2


Error in loading the data:  Ran out of input
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 09:04:21,016 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 3
2023-07-04 09:04:21,016 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 3
INFO:Worker_50151:Running the worker with id: 1 on iteration: 3


Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
